<a href="https://colab.research.google.com/github/anokhina-rgb/Consecutive-Translation-Trainer/blob/main/Scientometric_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import subprocess
import os
import zipfile
import numpy as np

# 1. Immediate Dependency Management
def install_dependencies():
    """Installs required packages and confirms imports."""
    packages = ["python-docx", "textblob", "matplotlib", "spacy", "numpy", "Pillow"]
    for pkg in packages:
        import_name = "docx" if pkg == "python-docx" else pkg
        try:
            __import__(import_name)
        except ImportError:
            print(f"Installing missing dependency: {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

    # Ensure spaCy model is present
    try:
        import spacy
        spacy.load("en_core_web_sm")
    except (ImportError, OSError):
        print("Downloading spaCy model...")
        subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])

install_dependencies()

# 2. Imports (Guaranteed safe after installation)
import spacy
from textblob import TextBlob
from docx import Document
import matplotlib.pyplot as plt
from google.colab import files

class ScientometricAnalyzer:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.criteria = [
            ("Структура роботи", 5), ("Оформлення", 10), ("Вступ", 10),
            ("Огляд літератури", 15), ("Практична частина", 20), ("Висновки", 10)
        ]

    def get_linguistic_profile(self, file_path):
        """Extracts quantitative linguistic metrics."""
        doc = Document(file_path)
        text = "\n".join([p.text for p in doc.paragraphs if len(p.text) > 20])
        nlp_doc = self.nlp(text)
        blob = TextBlob(text)

        return {
            "Complexity": len(nlp_doc) / max(len(list(nlp_doc.sents)), 1),
            "Tone": blob.sentiment.polarity,
            "Objectivity": 1 - blob.sentiment.subjectivity,
            "Structure": len([tok for tok in nlp_doc if tok.dep_ == "auxpass"]) / 10
        }

    def generate_radar_chart(self, metrics):
        """Generates and saves the linguistic radar chart as a PNG."""
        categories = list(metrics.keys())
        values = [min(v / (10 if k == "Complexity" else 1), 1) for k, v in metrics.items()]

        N = len(categories)
        angles = [n / float(N) * 2 * np.pi for n in range(N)]
        values += values[:1]; angles += angles[:1]

        fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
        ax.fill(angles, values, color='#264653', alpha=0.25)
        ax.plot(angles, values, color='#264653', linewidth=2)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(categories, fontsize=12)
        plt.title("Academic Linguistic Profile", size=15, y=1.1)
        plt.savefig("profile.png")
        plt.close()

    def run_full_pipeline(self):
        """Master Pipeline Execution."""
        print("--- Please upload your .docx file ---")
        uploaded = files.upload()
        file_path = list(uploaded.keys())[0]

        metrics = self.get_linguistic_profile(file_path)
        self.generate_radar_chart(metrics)

        # Word Report Generation
        doc = Document()
        doc.add_heading('Scientometric Analysis Report', 0)

        doc.add_heading('Quantitative Metrics', level=1)
        table = doc.add_table(rows=len(metrics)+1, cols=2)
        table.style = 'Table Grid'
        for i, (k, v) in enumerate(metrics.items()):
            table.cell(i,0).text = k
            table.cell(i,1).text = f"{v:.2f}"

        doc.add_heading('Formal 6-Point Evaluation Criteria', level=1)
        eval_table = doc.add_table(rows=len(self.criteria)+1, cols=2)
        eval_table.style = 'Table Grid'
        for i, (name, pts) in enumerate(self.criteria, 1):
            eval_table.cell(i,0).text = name
            eval_table.cell(i,1).text = f"___ / {pts}"

        doc.save("Formal_Evaluation.docx")

        # Final Bundle
        with zipfile.ZipFile("scientometric_research_bundle.zip", 'w') as zf:
            zf.write("Formal_Evaluation.docx")
            zf.write("profile.png")

        print("Pipeline complete. Downloading bundle...")
        files.download("scientometric_research_bundle.zip")

if __name__ == "__main__":
    analyzer = ScientometricAnalyzer()
    analyzer.run_full_pipeline()

Installing missing dependency: python-docx...
Installing missing dependency: Pillow...
--- Please upload your .docx file ---


Saving Курсова_Бут_Група 13_22_до друку.docx to Курсова_Бут_Група 13_22_до друку.docx
Pipeline complete. Downloading bundle...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>